In [1]:
import sys
sys.path.append('.')
import samna
import numpy as np
import matplotlib

matplotlib.use("TkAgg")          # Or "Qt5Agg", "MacOSX", "WebAgg"
import matplotlib.pyplot as plt
import samna.dynapse1 as dyn1

import dynapse1utils as ut
from netgen import Neuron, NetworkGenerator
from params_all_cores import *
import time
import importlib
from collections import deque
import threading

# sys.path.append('../Tools')

In [2]:
devices = samna.device.get_unopened_devices()
print(devices)

[device::DeviceInfo(serial_number=00000001, usb_bus_number=1, usb_device_address=4, logic_version=5, device_type_name=Dynapse1DevKit)]


In [3]:
#devices = samna.device.get_unopened_devices()
model   = samna.device.open_device(devices[int(0)])

In [4]:
api = model.get_dynapse1_api()
config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 
param_group_c1 = config1.chips[0].cores[1].parameter_group 
param_group_c2 = config1.chips[0].cores[2].parameter_group 
param_group_c3 = config1.chips[0].cores[3].parameter_group 

In [5]:
param_list = ["IF_AHTAU_N", "IF_AHTHR_N", "IF_AHW_P", "IF_BUF_P", "IF_DC_P", "IF_NMDA_N", "IF_RFR_N", "IF_TAU1_N", "IF_TAU2_N", "IF_THR_N", "NPDPIE_TAU_F_P", "NPDPIE_TAU_S_P", "NPDPIE_THR_F_P", "NPDPIE_THR_S_P", 
              "NPDPII_TAU_F_P", "NPDPII_TAU_S_P", "NPDPII_THR_F_P", "NPDPII_THR_S_P", "PS_WEIGHT_EXC_F_N", "PS_WEIGHT_EXC_S_N", "PS_WEIGHT_INH_F_N", "PS_WEIGHT_INH_S_N", "PULSE_PWLK_P", "R2R_P"]

In [6]:
# ----------------  stimulus: a Gaussian bump ----------------
n_pts     = 1000                 # number of samples
t_end     = 1.0                  # seconds  (→ dt = 1 ms)
t         = np.linspace(0, t_end, n_pts, endpoint=False)
x         = np.linspace(-4, 4, n_pts)
sigma = 0.6  # Try smaller values: 1.0 (default), 0.5, 0.25, etc.
gauss = (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * (x / sigma)**2)
#I_peak    = 30000000e-12              # 1000 pA
I_peak    = 30000000e-12              # 1000 pA

I         = gauss/gauss.max() * I_peak   # injected current (A)

# convert to pA for nicer y‑axis numbers
I_pA = I * 1e12                 # A → pA

# ---------------- Plot stimulus waveform ----------------
plt.figure()
plt.plot(t*1e3, I_pA)           # x‑axis in ms
plt.xlabel('Time (ms)')
plt.ylabel('Injected current (pA)')
plt.title('Gaussian current stimulus (σ = 0.6)')
plt.tight_layout()

# ---------------- Plot spike times ----------------
# reproduce spike detection (same loop as user)
tau_m, R_m = 20e-3, 100e6
C_m = tau_m / R_m
v_rest = v_reset = -65e-3
v_thresh = -50e-3
t_ref  = 2e-3
dt     = t_end / n_pts


# ----------------  LIF neuron parameters ----------------------
tau_m     = 20e-3                # 20 ms membrane time constant
R_m       = 100e6                # 100 MΩ  (=> C = tau/R)
C_m       = tau_m / R_m
v_rest    = -65e-3               # -65 mV
v_reset   = -65e-3
v_thresh  = -50e-3               # spike threshold
t_ref     = 2e-3                 # 2 ms refractory period
dt        = t_end / n_pts        # simulation time-step (s)

# ----------------  simulation loop ----------------------------
v        = v_rest
next_ok  = 0.0                   # time when refractory ends
v_trace  = np.empty(n_pts)
spikes   = []

for k in range(n_pts):
    if t[k] >= next_ok:          # not in refractory
        dv = (-(v - v_rest) + R_m * I[k]) / (R_m * C_m) * dt
        v += dv
        if v >= v_thresh:        # spike!
            spikes.append(t[k])
            v = v_reset
            next_ok = t[k] + t_ref
    v_trace[k] = v

spike_times_all = np.array(spikes)

spike_ids = np.full(len(spikes), 1)

spikegen_ids = [(0, 1, n) for n in range(10)]

# separate figure for raster‑like spike markers
plt.figure()
plt.eventplot(spike_times_all*1e3, orientation='horizontal', linelength=0.1)
plt.xlabel('Time (ms)')
plt.yticks([])
plt.title(f'Spike times generated by the stimulus ({len(spike_times_all)} spikes)')
plt.tight_layout()

plt.show()

2025-07-18 11:04:08.171 python[64963:44056186] +[IMKClient subclass]: chose IMKClient_Modern
2025-07-18 11:04:08.171 python[64963:44056186] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [7]:
eventsBuffer = deque(maxlen=500)

In [8]:
def collect_spikes(sink_node, runningFlag):
    while runningFlag[0]:  # Check first element of list
        eventsBuffer.extend(sink_node.get_events())

In [9]:
import params_all_cores

importlib.reload(params_all_cores)
config1 = model.get_configuration()

pop_nr = 4

# 1)  Declare the set of bad neurons once, in (chip, core, neuron_id) format

BROKEN_NEURONS = {(0, 1, 51), (0, 1, 71), (0, 1, 80), (0, 1, 88), (0, 1, 91)}          #  ⬅️  add more here if needed

def is_ok(chip: int, core: int, nid: int) -> bool:
    """True if this physical neuron should be used."""
    return (chip, core, nid) not in BROKEN_NEURONS

In [10]:
import importlib, dynapse1utils as ut

importlib.reload(params_all_cores)

importlib.reload(ut)

config1 = model.get_configuration()
param_group_c0 = config1.chips[0].cores[0].parameter_group 

p_E_E   = 1
p_mexican = 1
p_I_I   = 0 #1
p_E_I   = 0 #0.1
p_I_E   = 0 #0.4

# initiate network 
net_gen = NetworkGenerator()
net_gen.clear_network()

# create spikegens, one per ring attractor neural pop 
spikegen_ids = [(0, 0, n) for n in range(10)]
#isi_spikegen = Neuron(0, 0, 200, True)
#isi_neuron = Neuron(0, 1, 200)

spikegens = []
for spikegen_id in spikegen_ids:
    spikegens.append(Neuron(spikegen_id[0], spikegen_id[1], spikegen_id[2], True))

print(spikegens)

#spikegens.append(isi_spikegen)

# Create ring neuron populations
chip = 0
core = 1
npop = 4
NBINS = 10
offset_nr = 48

ring_pops = []
next_id = offset_nr
for _ in range(NBINS):
    pop = []
    while len(pop) < npop:
        if is_ok(chip, core, next_id):
            pop.append(Neuron(chip, core, next_id))
        next_id += 1
    ring_pops.append(pop)


# create inhibitory population that connects to all other pops
core_inh = 2
start_inh_neuron = 4
npop_inh = 4
pop_inhibitory = [Neuron(chip, core_inh, j) for j in range(start_inh_neuron, start_inh_neuron + npop_inh, 1)]
pop_inhibitory


# SPIKEGEN CONNECTIONS: one spike-gen (index i) permanently drives ring_pops[i]
for sg, pop in zip(spikegens, ring_pops):
    for neuron in pop:
        net_gen.add_connection(sg, neuron, dyn1.Dynapse1SynType.AMPA)
    
# connect isi spikegen to isi neuron 
#net_gen.add_connection(isi_spikegen, isi_neuron, dyn1.Dynapse1SynType.AMPA)

# self excitation in each neural population in the ring: (todo determine if this is needed) 
for pop in ring_pops:
    for pre in pop:
        for post in pop:
            if pre is not post and np.random.rand() < p_E_E:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.AMPA)
                                
# MEXICAN HAT CONNECTIONS
OFFSET_1 = (-1, 1)         
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_1:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)

OFFSET_2 = (-2, 2)          # ±3 bins wide “hat”

    # todo excitatory connections to second neighbors

for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_2:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)


    #  excitatory connections to third neighbors
OFFSET_3 = (-3, 3)          # ±3 bins wide “hat”
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_3:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.NMDA)
                print(post)

    # todo inhibitory connections to all of the other pops
OFFSET_inh = (-6, -5, -4, 4, 5, 6)          # all of the pops that are not being excited
for i, pop_i in enumerate(ring_pops):
    for pre in pop_i:
        for d in OFFSET_inh:
            j = (i + d) % NBINS          # wrap around
            for post in ring_pops[j]:
                net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)
                #net_gen.add_connection(pre, post, dyn1.Dynapse1SynType.GABA_B)

# INH → EXC  (global inhibition pop to all pops in the ring)
for inh in pop_inhibitory:
    for pop in ring_pops:
        for exc in pop:
            net_gen.add_connection(inh, exc, dyn1.Dynapse1SynType.GABA_B)

# EXC → INH  (drive the global inhibition pop from all pops in the ring)
for pop in ring_pops:
    for exc in pop:
        for inh in pop_inhibitory:
            net_gen.add_connection(exc, inh, dyn1.Dynapse1SynType.NMDA)


# make a dynapse1config using the network
new_config = net_gen.make_dynapse1_configuration()

# apply the configuration
model.apply_configuration(new_config)

# Set hardware parameters
set_params(model)

fpga_spike_gen = model.get_fpga_spike_gen() # set FPGA 


monitored_neurons = [
    (n.chip_id, n.core_id, n.neuron_id)
    for pop in ring_pops
    for n   in pop
]

monitored_neurons.extend([
    (neuron.chip_id, neuron.core_id, neuron.neuron_id)
    for neuron in pop_inhibitory  
])


graph, filter_node, sink_node = ut.create_neuron_select_graph(model, monitored_neurons)
graph.start()

# clear the buffer
sink_node.get_events()

# select the neurons to monitor
filter_node.set_neurons(monitored_neurons)

api.reset_timestamp()

ut.set_neuron_tau1(model, 0, 0, (7, 255))
ut.set_neuron_tau1(model, 0, 1, (7, 255))
ut.set_neuron_tau1(model, 0, 2, (7, 255))
ut.set_neuron_tau1(model, 0, 3, (7, 255))

time.sleep(1)

ut.set_neuron_tau1(model, 0, 0, (4, 50))
ut.set_neuron_tau1(model, 0, 1, (4, 50))
ut.set_neuron_tau1(model, 0, 2, (4, 50))
ut.set_neuron_tau1(model, 0, 3, (4, 50))


spike_ids_all = spike_ids

# Sort input events in time order
sort_indices = np.argsort(spike_times_all)
all_spike_times = spike_times_all[sort_indices]
all_spike_ids = spike_ids_all[sort_indices]

current_pop = {'value': 5}        # start with pop 0
spike_ids   = np.full(len(all_spike_times), current_pop['value'])

ut.set_fpga_spike_gen(
    fpga_spike_gen,
    all_spike_times,
    #all_spike_ids,
    spike_ids,
    #target_chips=[0] * len(all_spike_ids),
    target_chips=[0] * len(spike_ids),
    isi_base=900,
    repeat_mode=False)

fpga_spike_gen.start()

[C0c0s0, C0c0s1, C0c0s2, C0c0s3, C0c0s4, C0c0s5, C0c0s6, C0c0s7, C0c0s8, C0c0s9]
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n78
C0c1n79
C0c1n81
C0c1n82
C0c1n61
C0c1n62
C0c1n63
C0c1n64
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n83
C0c1n84
C0c1n85
C0c1n86
C0c1n65
C0c1n66
C0c1n67
C0c1n68
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n87
C0c1n89
C0c1n90
C0c1n92
C0c1n69
C0c1n70
C0c1n72
C0c1n73
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50
C0c1n52
C0c1n74
C0c1n75
C0c1n76
C0c1n77
C0c1n48
C0c1n49
C0c1n50

In [11]:
# ────────────────────────────────────── #
# ======== KDE-style smoothing =========
# ────────────────────────────────────── #
import scipy.ndimage as ndi

def smooth_rates_circular(counts, sigma_bins=1.0):
    """
    Gaussian-smooth an array living on a circular ring.
    counts      : 1-D numpy array of length NBINS
    sigma_bins  : std-dev of the Gaussian, expressed in *bin* units
    """
    # Pad three bins on each side so the wrap-around is seamless
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')
    return smoothed[3:-3]


In [15]:
# ────────────────────────────────
# 1-second stimulation test
# ────────────────────────────────

"""Stimulate one population after another with the stimulus defined above, then measure drift after 1 s"""

stim_interval = 1.0            # seconds between populations
errors_2A = []

# Extra lists for the raster
events_time_2A, events_id_2A = [], []

# Pre-compute a neuron-ID → pop-index lookup for speed
nid_to_pop = {n.neuron_id: pop_idx
              for pop_idx, pop in enumerate(ring_pops)
              for n in pop}

for target_pop in range(NBINS):
    # ── Re-program the FPGA spike generator ──
    spike_ids[:] = target_pop
    ut.set_fpga_spike_gen(fpga_spike_gen,
                          all_spike_times, spike_ids,
                          target_chips=[0]*len(spike_ids),
                          isi_base=900, repeat_mode=False)

    eventsBuffer.clear()       # fresh slate
    fpga_spike_gen.start()

    # ── Collect events for 'stim_interval' ──
    t0 = time.time()
    while time.time() - t0 < stim_interval:
        time.sleep(0.01)
    fpga_spike_gen.stop()
    eventsBuffer.extend(sink_node.get_events())       # grab the tail end


    # ── Compute population firing counts & stash events ──
    rates = np.zeros(NBINS)
    for e in list(eventsBuffer):
        pop_idx = nid_to_pop.get(e.neuron_id, None)
        if pop_idx is not None:
            rates[pop_idx] += 1
        # save for raster (store relative time to this sweep’s start)
        events_time_2A.append(e.timestamp*1e-6)
        events_id_2A.append(e.neuron_id)

    # ── Evaluate localisation error ──
    smoothed = smooth_rates_circular(rates, sigma_bins=1.0)
    max_pop  = int(np.argmax(smoothed))
    error    = (max_pop - target_pop) % NBINS
    if error > NBINS/2:
        error -= NBINS
    errors_2A.append(error)
    print(f"Stim {target_pop:2d} → peak {max_pop:2d}  (error {error:+d} bins)")

print("\nMean |error| (bins) for 1-s protocol:", np.mean(np.abs(errors_2A)))

# --- drift bar plot ---
plt.figure(figsize=(6, 4))
plt.bar(range(NBINS), errors_2A, width=0.8,
        color=['tab:red' if e < 0 else 'tab:blue' for e in errors_2A])
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Target population (0 – 9)')
plt.ylabel('Drift (bins)')
plt.title('Ring-attractor drift after 1-s stimulation\n(+ = clockwise, – = counter-clockwise)')
plt.xticks(range(NBINS))
plt.tight_layout()

# --- raster plot for Task 2A ---
plt.figure(figsize=(10, 4))
plt.scatter(events_time_2A, events_id_2A, s=4, alpha=0.6)
plt.xlabel('Time (s)')
plt.ylabel('Neuron ID')
plt.title('Raster plot – Task 2A (1-s sweep)')
plt.tight_layout()
plt.show()

Stim  0 → peak  8  (error -2 bins)
Stim  1 → peak  1  (error +0 bins)
Stim  2 → peak  1  (error -1 bins)
Stim  3 → peak  1  (error -2 bins)
Stim  4 → peak  3  (error -1 bins)
Stim  5 → peak  3  (error -2 bins)
Stim  6 → peak  5  (error -1 bins)
Stim  7 → peak  6  (error -1 bins)
Stim  8 → peak  7  (error -1 bins)
Stim  9 → peak  8  (error -1 bins)

Mean |error| (bins) for 1-s protocol: 1.2


In [12]:
nid_to_pop = {}
for pop_idx, pop in enumerate(ring_pops):
    for n in pop:
        nid_to_pop[n.neuron_id] = pop_idx          # 0-9

# give the inhibitory pop its own index (10) or any sentinel
INH_IDX = NBINS
for n in pop_inhibitory:
    nid_to_pop[n.neuron_id] = INH_IDX


In [ ]:
# ────────────────────────────────
# 5-second stimulation test
# ────────────────────────────────

import matplotlib.cm as cm
import matplotlib.patches as mpatches    # ← add this

"""Stimulate one population after another with the stimulus defined above, then measure drift after 1 s"""

stim_interval = 5.0            # seconds between populations
errors_2B = []
stim_markers_2B              = []  
t_onsets = []               # will hold the 10×15 start times


# Extra lists for the raster
events_time_2B, events_id_2B = [], []

for target_pop in range(NBINS):
    spike_ids[:] = target_pop
    ut.set_fpga_spike_gen(fpga_spike_gen,
                          all_spike_times, spike_ids,
                          target_chips=[0]*len(spike_ids),
                          isi_base=900, repeat_mode=False)

    eventsBuffer.clear()
    
    t_onsets.append(time.time())    # wall-clock seconds

    fpga_spike_gen.start()
    
    first_evt_ts = None                    # will hold first-spike timestamp


    t0 = time.time()

    
    while time.time() - t0 < stim_interval:
        new_evts = sink_node.get_events()
        if new_evts and first_evt_ts is None:
            first_evt_ts = new_evts[0].timestamp   # µs
        eventsBuffer.extend(new_evts)
        time.sleep(0.05)
    fpga_spike_gen.stop()
    eventsBuffer.extend(sink_node.get_events())

    # ── keep the first-spike time as the onset marker ──
    if first_evt_ts is not None:
        stim_markers_2B.append(first_evt_ts * 1e-6)   # µs → s
        
    ONE_SEC = 1_000_000     # µs
    
    # ---------- replace t_end / window_start ----------
    if eventsBuffer:
        t_end = eventsBuffer[-1].timestamp      # newest event
    else:
        t_end = 0                               # no events? unlikely
    window_start = t_end - ONE_SEC

    recent = [e for e in eventsBuffer if e.timestamp >= window_start]
    
    INH_IDX = NBINS          # 10

    rates = np.zeros(NBINS)
    
    #for e in list(eventsBuffer):
    for e in list(recent):
        pop_idx = nid_to_pop.get(e.neuron_id, None)
        
        if pop_idx is None or pop_idx == INH_IDX:
            continue
        
        rates[pop_idx] += 1


        # save for raster
        events_time_2B.append(e.timestamp*1e-6)
        events_id_2B.append(e.neuron_id)

    smoothed = smooth_rates_circular(rates, sigma_bins=1.0)
    max_pop  = int(np.argmax(smoothed))
    error    = (max_pop - target_pop) % NBINS
    if error > NBINS/2:
        error -= NBINS
    errors_2B.append(error)
    print(f"Stim {target_pop:2d} → peak {max_pop:2d}  (error {error:+d} bins)")
    
    # ◀︎ PRINT SPIKES PER POP HERE ▶︎
    print("  spikes per pop (last 1 s):", ["%2d"%r for r in rates])

print("\nMean |error| (bins) for 5-s protocol:", np.mean(np.abs(errors_2B)))

print("how many markers?", len(stim_markers_2B))
print("marker times (s):", stim_markers_2B[:5])
print("scatter x-range :", min(events_time_2B), "→", max(events_time_2B))

# -----------------------------------------
# --- drift bar plot  (signed) ------------
# -----------------------------------------
plt.figure(figsize=(6, 4))
col = ['tab:red' if e < 0 else 'tab:blue' for e in errors_2B]

plt.bar(range(NBINS), errors_2B, width=0.8, color=col)
plt.axhline(0, color='black', linewidth=0.8)

plt.xlabel('Target population (0 – 9)')
plt.ylabel('Drift (bins)  (+ = clockwise, – = anticlockwise)')
plt.title('Ring-attractor drift after 5-s stimulation')
plt.xticks(range(NBINS))
plt.tight_layout()

# -----------------------------------------
# --- raster plot  ------------------------
# -----------------------------------------
import matplotlib.cm as cm

# tab10 gives 10 distinct colours – one per population
pop_colors = cm.get_cmap('tab10', NBINS+1)   # 11 colours

# Map every event → its population → a RGBA colour
evt_colors = []
for eid in events_id_2B:
    pop_idx = nid_to_pop.get(eid, None)
    if pop_idx is None:               # not part of ring
        evt_colors.append('lightgrey')
    else:
        evt_colors.append(pop_colors(pop_idx))

plt.figure(figsize=(10, 4))
plt.scatter(events_time_2B, events_id_2B,
            s=4, c=evt_colors, alpha=0.6, marker='o')

# dashed guide lines at pop boundaries (optional):
for pop in range(NBINS):
    ids = [n.neuron_id for n in ring_pops[pop]]
    if ids:
        y_min, y_max = min(ids)-0.5, max(ids)+0.5
        plt.axhline(y_min, color='grey', linestyle='--', linewidth=0.3)
        plt.text(events_time_2B[0], (y_min+y_max)/2, f'pop {pop}',
                 va='center', ha='left', fontsize=8, color=pop_colors(pop))

# vertical onset markers
#for t in stim_markers_2B:
    #plt.axvline(t, linestyle='--', linewidth=0.8, color='red', alpha=0.4)
    


# ── ADD LEGEND HANDLES ──
handles = [
    mpatches.Patch(color=pop_colors(i), label=f'pop {i}')
    for i in range(NBINS)
]
handles.append(mpatches.Patch(color='lightgrey', label='other/inhibitory'))

plt.legend(handles=handles,
           bbox_to_anchor=(1.02, 1),   # just outside right
           loc='upper left',
           borderaxespad=0.)

plt.xlabel('Time (s)')
plt.ylabel('Neuron ID')
plt.title('Raster – 5-s stimulation (coloured by population)')
plt.tight_layout()
plt.show()


Stim  0 → peak  0  (error +0 bins)
  spikes per pop (last 1 s): [' 4', ' 5', ' 2', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0', ' 4']
Stim  1 → peak  1  (error +0 bins)
  spikes per pop (last 1 s): [' 0', ' 1', ' 1', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0']
Stim  2 → peak  1  (error -1 bins)
  spikes per pop (last 1 s): [' 5', ' 4', ' 4', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0', ' 3']
Stim  3 → peak  0  (error -3 bins)
  spikes per pop (last 1 s): [' 5', ' 5', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0', ' 4', ' 5']
Stim  4 → peak  8  (error +4 bins)
  spikes per pop (last 1 s): [' 3', ' 0', ' 0', ' 0', ' 0', ' 0', ' 0', ' 4', ' 4', ' 4']
Stim  5 → peak  5  (error +0 bins)
  spikes per pop (last 1 s): [' 0', ' 0', ' 0', ' 0', ' 3', ' 6', ' 4', ' 2', ' 0', ' 0']
Stim  6 → peak  5  (error -1 bins)
  spikes per pop (last 1 s): [' 0', ' 0', ' 0', ' 0', ' 2', ' 6', ' 3', ' 1', ' 0', ' 0']
Stim  7 → peak  5  (error -2 bins)
  spikes per pop (last 1 s): [' 0', ' 0', ' 0', ' 0', ' 4', ' 4', ' 4', ' 2', ' 0', ' 0']


/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_79010/1798272470.py:147: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  pop_colors = cm.get_cmap('tab10', NBINS+1)   # 11 colours
2025-07-09 10:57:59.845 python[79010:26670334] The class 'NSSavePanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


In [42]:
# randomised test 5 s stimulation 

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import time

# --- parameters ---
NBINS      = 10
N_REPS     = 1
STIM_US    = 5_000_000         # 5 s in µs
ANALYSIS_US= 1_000_000         # last 1 s in µs
INH_IDX    = NBINS             # inhibitory pop index

# prepare storage
rate_profiles = {pop: [] for pop in range(NBINS)}  # full vectors
errors_by_pop  = {pop: [] for pop in range(NBINS)}

events_time = []
events_id   = []
stim_markers= []

# randomised sweep
for rep in range(N_REPS):
    order = np.random.permutation(NBINS)
    for target_pop in order:
        # reprogram the FPGA
        spike_ids[:] = target_pop
        ut.set_fpga_spike_gen(fpga_spike_gen,
                              all_spike_times, spike_ids,
                              target_chips=[0]*len(spike_ids),
                              isi_base=900,
                              repeat_mode=False)
        eventsBuffer.clear()
        fpga_spike_gen.start()

        # mark the moment of first delivery
        t0_chip = None
        while t0_chip is None:
            new = sink_node.get_events()
            if new:
                t0_chip = new[0].timestamp
                stim_markers.append(t0_chip * 1e-6)
            eventsBuffer.extend(new)
            time.sleep(0.01)

        # collect exactly 5 s of chip time
        t_start = t0_chip
        while True:
            now = sink_node.get_events()
            eventsBuffer.extend(now)
            if eventsBuffer and eventsBuffer[-1].timestamp - t_start >= STIM_US:
                break
            time.sleep(0.01)

        fpga_spike_gen.stop()
        eventsBuffer.extend(sink_node.get_events())

        # extract last-1 s window
        t_end        = eventsBuffer[-1].timestamp
        window_start = t_end - ANALYSIS_US
        recent = [e for e in eventsBuffer if e.timestamp >= window_start]

        # count per bin, skip inh
        counts = np.zeros(NBINS)
        for e in recent:
            pop_idx = nid_to_pop.get(e.neuron_id)
            if pop_idx is None or pop_idx == INH_IDX:
                continue
            counts[pop_idx] += 1

            # stash for raster
            events_time.append(e.timestamp * 1e-6)
            events_id.append(e.neuron_id)

        # smooth & pick peak
        smoothed = smooth_rates_circular(counts, sigma_bins=1.0)
        peak     = int(np.argmax(smoothed))
        error    = (peak - target_pop) % NBINS
        if error > NBINS/2: error -= NBINS

        rate_profiles[target_pop].append(smoothed)
        errors_by_pop[target_pop].append(error)

# --- 1) Bar-plots of full profiles per population ---
fig, axs = plt.subplots(2, 5, figsize=(12, 6), sharey=True)
for pop, ax in zip(range(NBINS), axs.ravel()):
    # average across reps
    avg_profile = np.mean(rate_profiles[pop], axis=0)
    ax.bar(np.arange(NBINS), avg_profile, width=0.8)
    ax.set_title(f'Pop {pop}')
    ax.set_xticks(range(NBINS))
    if pop % 5 == 0:
        ax.set_ylabel('Firing count (last 1 s)')
fig.suptitle('Average activity profiles after 5 s stim (15 reps)')
plt.tight_layout(rect=[0,0,1,0.95])

# --- 2) Signed drift bar-plot (mean over reps) ---
mean_errors = [np.mean(errors_by_pop[p]) for p in range(NBINS)]
cols        = ['tab:red' if e < 0 else 'tab:blue' for e in mean_errors]
plt.figure(figsize=(6,4))
plt.bar(range(NBINS), mean_errors, color=cols)
plt.axhline(0, color='k', lw=0.8)
plt.xlabel('Target population')
plt.ylabel('Mean drift (bins) (+ = CW, – = CCW)')
plt.title('Drift after 5 s stim (signed)')
plt.tight_layout()

# --- 3) Raster with legend outside ---
# map neuron IDs → pop again (for colouring & legend)
pop_colors = cm.get_cmap('tab10', NBINS)
evt_colors = []
for nid in events_id:
    p = nid_to_pop.get(nid)
    evt_colors.append(pop_colors(p) if p is not None and p<NBINS else 'lightgrey')

plt.figure(figsize=(10,4))
plt.scatter(events_time, events_id, s=4, c=evt_colors, alpha=0.6)
for t in stim_markers:
    plt.axvline(t, ls='--', lw=0.8, c='red', alpha=0.4)
plt.xlabel('Time (s)')
plt.ylabel('Neuron ID')
plt.title('Raster – randomised 5 s stimuli')

# legend handles
handles = [mpatches.Patch(color=pop_colors(i), label=f'Pop {i}')
           for i in range(NBINS)]
handles.append(mpatches.Patch(color='lightgrey', label='Other/Inh'))

plt.legend(handles=handles,
           bbox_to_anchor=(1.02, 1),
           loc='upper left',
           borderaxespad=0.)
plt.tight_layout()

plt.show()   # uncomment if running as a script


KeyboardInterrupt: 

In [12]:
import scipy.ndimage as ndi
def smooth_rates_circular(counts, sigma_bins=1.0):
    padded = np.r_[counts[-3:], counts, counts[:3]]
    smoothed = ndi.gaussian_filter1d(padded, sigma=sigma_bins, mode='wrap')

    return smoothed[3:-3]

In [13]:
# Start spike collection in a thread
running_flag = [True]  # Use list for mutable flag
spike_thread = threading.Thread(target=collect_spikes, args=(sink_node, running_flag))
spike_thread.daemon = True
spike_thread.start()
# animation = start_spike_visualization(eventsBuffer)

In [ ]:
def on_key(event):
    k = event.key
    if k.isdigit():                       # 0-9 → choose new population
        new = int(k)
        if 0 <= new < NBINS:
            current_pop['value'] = new
            fpga_spike_gen.stop()         # reload the stimuli for that pop
            spike_ids[:] = new            # update *in-place* so length stays the same
            
            """ut.set_neuron_tau1(model, 0, 0, (7, 255))
            ut.set_neuron_tau1(model, 0, 1, (7, 255))
            ut.set_neuron_tau1(model, 0, 2, (7, 255))
            ut.set_neuron_tau1(model, 0, 3, (7, 255))

            time.sleep(1)

            ut.set_neuron_tau1(model, 0, 0, (4, 50))
            ut.set_neuron_tau1(model, 0, 1, (4, 50))
            ut.set_neuron_tau1(model, 0, 2, (4, 50))
            ut.set_neuron_tau1(model, 0, 3, (4, 50))"""
            
            ut.set_fpga_spike_gen(fpga_spike_gen,
                                   all_spike_times,
                                   spike_ids,
                                   target_chips=[0]*len(spike_ids),
                                   isi_base=900,
                                   repeat_mode=False)
            fpga_spike_gen.start()
            print(f"→ Stimulating pop {new}")
    elif k == 'q':                        # quit cleanly
        running_flag[0] = False


# Set up interactive plotting
plt.ion()

# Create figure with two subplots side by side
fig, (ax_raster, ax_rate) = plt.subplots(1, 2, figsize=(15, 6))
fig.canvas.mpl_connect('key_press_event', on_key)   # moved ↓ here
fig.show()                      # ← opens ONE browser tab

# Setup raster plot in first subplot
scatter = ax_raster.scatter([], [], s=10, alpha=0.6)
xlim_max = 10
ax_raster.set_xlabel('Time (s)')
ax_raster.set_ylabel('Neuron Index')
ax_raster.set_title('Dynap-SE1 Ring Attractor Spikes')

# Setup firing rate profile in second subplot
# Create positions for neurons (0 to 2π for the ring)
positions = np.linspace(0, 2*np.pi, NBINS, endpoint=False)

# dashed guide lines at every population centre
for p in positions:
    ax_rate.axvline(p,
                    linestyle='--',
                    linewidth=0.8,
                    color='gray',
                    alpha=0.4)

# optional: label each line with the pop index
for idx, p in enumerate(positions):
    ax_rate.text(p,            ax_rate.get_ylim()[1]*1.02,
                 str(idx),
                 ha='center', va='bottom', fontsize=9)

rate_line, = ax_rate.plot(positions, np.zeros_like(positions), 'o-', markersize=8)
ax_rate.set_xlim(0, 2*np.pi)
ax_rate.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax_rate.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
ax_rate.set_xlabel('Position (radians)')
ax_rate.set_ylabel('Firing rate (Hz)')
ax_rate.set_title('Firing Rate Profile')
ax_rate.grid(True, alpha=0.3)

plt.tight_layout()

# Variables for rate calculation
window_size = 1.0  # seconds - time window for calculating rates
last_update_time = 0

# Real-time plotting loop
while running_flag[0]:
    try:
        if len(eventsBuffer) > 0:
            # Extract spike data for raster plot
            spikesID = [e.neuron_id for e in eventsBuffer]
            spikesTimes = [e.timestamp*1e-6 for e in eventsBuffer]  # Convert to seconds
            
            print(spikesTimes)
            print(spikesID)
            
            if not spikesID:
                continue
                
            # Update raster plot
            ax_raster.set_ylim(min(spikesID) - 0.5, max(spikesID) + 0.5)
            scatter.set_offsets(np.column_stack((spikesTimes, spikesID)))
            
            # Update time window
            current_time = max(spikesTimes)
            ax_raster.set_xlim(max(0, current_time - xlim_max), current_time + 1)
            ax_raster.set_ylim(0, 95)
            
            # Calculate firing rates across the ring (using recent time window)
            window_start = current_time - window_size
            recent_events = [e for e in eventsBuffer if e.timestamp*1e-6 >= window_start]
            
            # Count spikes for each bin in the ring
            firing_rates = np.zeros(NBINS)
            
            # Map neuron IDs to their bin/position in the ring
            for event in recent_events:
                neuron_id = event.neuron_id
                for i, pop in enumerate(ring_pops):
                    pop_ids = [n.neuron_id for n in pop]
                    if neuron_id in pop_ids:
                        firing_rates[i] += 1
                        break
                    
            # numpy histogram for firing rate then divide by bins
            
            # Convert to Hz (spikes per second)
            firing_rates = firing_rates / window_size
            
            # Update firing rate plot
            smoothed_rates = smooth_rates_circular(firing_rates, sigma_bins=1.0)
            rate_line.set_ydata(smoothed_rates)
            max_rate = max(smoothed_rates) if any(smoothed_rates > 0) else 10
            
            #ax_rate.set_ylim(0, 20)  

            
            # Refresh both plots
            fig.canvas.draw_idle()
            fig.canvas.flush_events()   # keeps the websocket alive
            time.sleep(0.01)            # tiny CPU-friendly sleep

            #plt.pause(0.0001)  # Shorter pause for smoother updates
            
    except Exception as e:
        print(f"Plotting error: {e}")
        import traceback
        traceback.print_exc()  # Print detailed error information
        break

/var/folders/38/x2v4gv396nz13mws24nz37sw0000gn/T/ipykernel_64963/4119013336.py:76: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


→ Stimulating pop 5
→ Stimulating pop 9


In [ ]:
# To stop everything cleanly:
running_flag[0] = False
spike_thread.join(timeout=1.0)
plt.ioff()
plt.close()